<a href="https://colab.research.google.com/github/k2-fsa/colab/blob/master/aed_decoding_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 介绍

本 colab notebook 详细演示如何用一个 attention encoder-decoder 模型识别一个音频文件。

# 下载模型和测试音频

使用 https://k2-fsa.github.io/sherpa/onnx/nemo/canary.html#sherpa-onnx-nemo-canary-180m-flash-en-es-de-fr-int8-english-spanish-german-french

In [ ]:
%%shell

wget https://github.com/k2-fsa/sherpa-onnx/releases/download/asr-models/sherpa-onnx-nemo-canary-180m-flash-en-es-de-fr-int8.tar.bz2
tar xvf sherpa-onnx-nemo-canary-180m-flash-en-es-de-fr-int8.tar.bz2
rm sherpa-onnx-nemo-canary-180m-flash-en-es-de-fr-int8.tar.bz2


--2025-10-12 12:19:40--  https://github.com/k2-fsa/sherpa-onnx/releases/download/asr-models/sherpa-onnx-nemo-canary-180m-flash-en-es-de-fr-int8.tar.bz2
Resolving github.com (github.com)... 140.82.112.4
Connecting to github.com (github.com)|140.82.112.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/531380835/34786345-d0f4-403b-b031-03361fd30c5d?sp=r&sv=2018-11-09&sr=b&spr=https&se=2025-10-12T13%3A19%3A28Z&rscd=attachment%3B+filename%3Dsherpa-onnx-nemo-canary-180m-flash-en-es-de-fr-int8.tar.bz2&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2025-10-12T12%3A18%3A44Z&ske=2025-10-12T13%3A19%3A28Z&sks=b&skv=2018-11-09&sig=phaUaOLjzWG4ATqO9UgVVU7947LaHaKJJ5%2FtF144aow%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5M

In [ ]:
%%shell

ls -lh sherpa-onnx-nemo-canary-180m-flash-en-es-de-fr-int8/test_wavs/en.wav


-rw-r--r-- 1 501 staff 181K Jul  7 08:03 sherpa-onnx-nemo-canary-180m-flash-en-es-de-fr-int8/test_wavs/en.wav


我们要用到其中的
 - encoder.int8.onnx
 - decoder.int8.onnx
 - tokens.txt
 - test_wavs/en.wav

# 安装依赖

In [ ]:
%%shell

pip install onnxruntime soundfile kaldi-native-fbank

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.0/322.0 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.6 MB/s eta 0:00:00


# 加载模型

In [ ]:
import onnxruntime as ort
session_opts = ort.SessionOptions()
session_opts.inter_op_num_threads = 1
session_opts.intra_op_num_threads = 1


encoder = ort.InferenceSession(
    "./sherpa-onnx-nemo-canary-180m-flash-en-es-de-fr-int8/encoder.int8.onnx",
    sess_options=session_opts,
    providers=["CPUExecutionProvider"],
)

decoder = ort.InferenceSession(
    "./sherpa-onnx-nemo-canary-180m-flash-en-es-de-fr-int8/decoder.int8.onnx",
    sess_options=session_opts,
    providers=["CPUExecutionProvider"],
)

# 显示模型的输入和输出


In [ ]:
for i in encoder.get_inputs():
    print(i)

print("-----")

for i in encoder.get_outputs():
    print(i)

NodeArg(name='x', type='tensor(float)', shape=['N', 'T', 128])
NodeArg(name='x_len', type='tensor(int64)', shape=['N'])
-----
NodeArg(name='enc_states', type='tensor(float)', shape=['N', 'T', 1024])
NodeArg(name='enc_len', type='tensor(int64)', shape=['N'])
NodeArg(name='enc_mask', type='tensor(bool)', shape=['N', 'T'])


In [ ]:
for i in decoder.get_inputs():
    print(i)

print("-----")

for i in decoder.get_outputs():
    print(i)

NodeArg(name='decoder_input_ids', type='tensor(int32)', shape=[1, '2'])
NodeArg(name='decoder_mems_list_0', type='tensor(float)', shape=[1, 's1', 1024])
NodeArg(name='decoder_mems_list_1', type='tensor(float)', shape=[1, 's2', 1024])
NodeArg(name='decoder_mems_list_2', type='tensor(float)', shape=[1, 's3', 1024])
NodeArg(name='decoder_mems_list_3', type='tensor(float)', shape=[1, 's4', 1024])
NodeArg(name='decoder_mems_list_4', type='tensor(float)', shape=[1, 's5', 1024])
NodeArg(name='decoder_mems_list_5', type='tensor(float)', shape=[1, 's6', 1024])
NodeArg(name='enc_states', type='tensor(float)', shape=[1, 's7', 1024])
NodeArg(name='enc_mask', type='tensor(bool)', shape=[1, 's7'])
-----
NodeArg(name='logits', type='tensor(float)', shape=[1, 1, 5248])
NodeArg(name='next_decoder_mem_list_0', type='tensor(float)', shape=[1, 's1 + 1', 1024])
NodeArg(name='next_decoder_mem_list_1', type='tensor(float)', shape=[1, 's2 + 1', 1024])
NodeArg(name='next_decoder_mem_list_2', type='tensor(float

# 读取测试音频

In [ ]:
import soundfile as sf
import numpy as np
import librosa

def load_audio(filename):
    data, sample_rate = sf.read(
        filename,
        always_2d=True,
        dtype="float32",
    )
    if sample_rate != 16000:
        data = librosa.resample(
            data,
            orig_sr=sample_rate,
            target_sr=16000,
        )

    data = data[:, 0]  # use only the first channel
    samples = np.copy(np.ascontiguousarray(data))
    return samples

In [ ]:
from IPython.display import Audio, display

display(Audio("./sherpa-onnx-nemo-canary-180m-flash-en-es-de-fr-int8/test_wavs/en.wav", autoplay=True))

# 提取特征

In [ ]:
import kaldi_native_fbank as knf
import numpy as np

def create_fbank():
    opts = knf.FbankOptions()
    opts.frame_opts.dither = 0
    opts.frame_opts.remove_dc_offset = False
    opts.frame_opts.window_type = "hann"

    opts.mel_opts.low_freq = 0
    opts.mel_opts.num_bins = 128

    opts.mel_opts.is_librosa = True

    fbank = knf.OnlineFbank(opts)
    return fbank


def compute_features(audio) -> np.ndarray:
    fbank = create_fbank()
    assert len(audio.shape) == 1, audio.shape
    fbank.accept_waveform(16000, audio)
    ans = []
    processed = 0
    while processed < fbank.num_frames_ready:
        ans.append(np.array(fbank.get_frame(processed)))
        processed += 1
    features = np.stack(ans)
    mean = features.mean(axis=0, keepdims=True)
    stddev = features.std(axis=0, keepdims=True) + 1e-5
    features = (features - mean) / stddev
    return features

# 运行 encoder 模型

In [ ]:
samples = load_audio('./sherpa-onnx-nemo-canary-180m-flash-en-es-de-fr-int8/test_wavs/en.wav')
features = compute_features(samples)
print(features.shape)

encoder_out, encoder_masks = encoder.run(
    [
        encoder.get_outputs()[0].name,
        encoder.get_outputs()[2].name,
    ],
    {
        encoder.get_inputs()[0].name: features[None],
        encoder.get_inputs()[1].name: np.array([features.shape[0]], dtype=np.int64),
    },
)

(575, 128)


In [ ]:
print(encoder_out.shape, encoder_masks.shape)

(1, 72, 1024) (1, 72)


# 运行 decoder 模型

## 加载 tokens

In [ ]:
id2token = dict()
token2id = dict()
with open("./sherpa-onnx-nemo-canary-180m-flash-en-es-de-fr-int8/tokens.txt", encoding="utf-8") as f:
  for line in f:
      fields = line.split()
      if len(fields) == 2:
          t, idx = fields[0], int(fields[1])
          if line[0] == " ":
              t = " " + t
      else:
          t = " "
          idx = int(fields[0])

      id2token[idx] = t
      token2id[t] = idx

In [ ]:
decoder_input_ids = []
decoder_input_ids.append(token2id["<|startofcontext|>"])
decoder_input_ids.append(token2id["<|startoftranscript|>"])
decoder_input_ids.append(token2id["<|emo:undefined|>"])

decoder_input_ids.append(token2id[f"<|en|>"])
decoder_input_ids.append(token2id[f"<|en|>"])

decoder_input_ids.append(token2id[f"<|pnc|>"])

decoder_input_ids.append(token2id[f"<|noitn|>"])
decoder_input_ids.append(token2id["<|notimestamp|>"])
decoder_input_ids.append(token2id["<|nodiarize|>"])

states = [np.zeros((1, 0, 1024), dtype=np.float32) for _ in range(6)]

for pos, decoder_input_id in enumerate(decoder_input_ids):
  (logits, *new_states) = decoder.run(
            [
                decoder.get_outputs()[0].name,
                decoder.get_outputs()[1].name,
                decoder.get_outputs()[2].name,
                decoder.get_outputs()[3].name,
                decoder.get_outputs()[4].name,
                decoder.get_outputs()[5].name,
                decoder.get_outputs()[6].name,
            ],
            {
                decoder.get_inputs()[0].name: np.array([[decoder_input_id, pos]], dtype=np.int32),
                decoder.get_inputs()[1].name: states[0],
                decoder.get_inputs()[2].name: states[1],
                decoder.get_inputs()[3].name: states[2],
                decoder.get_inputs()[4].name: states[3],
                decoder.get_inputs()[5].name: states[4],
                decoder.get_inputs()[6].name: states[5],
                decoder.get_inputs()[7].name: encoder_out,
                decoder.get_inputs()[8].name: encoder_masks,
            },
  )
  states = new_states

In [ ]:
current = logits.argmax()
print(current)

1251


In [ ]:
ids = []
current = logits.argmax()

eos = token2id["<|endoftext|>"]
print('eos', eos)

pos = 1
for i in range(1, 200):
  if current == eos:
    break
  ids.append(current)

  decoder_input_ids = [current, i]
  (logits, *new_states) = decoder.run(
            [
                decoder.get_outputs()[0].name,
                decoder.get_outputs()[1].name,
                decoder.get_outputs()[2].name,
                decoder.get_outputs()[3].name,
                decoder.get_outputs()[4].name,
                decoder.get_outputs()[5].name,
                decoder.get_outputs()[6].name,
            ],
            {
                decoder.get_inputs()[0].name: np.array([decoder_input_ids], dtype=np.int32),
                decoder.get_inputs()[1].name: states[0],
                decoder.get_inputs()[2].name: states[1],
                decoder.get_inputs()[3].name: states[2],
                decoder.get_inputs()[4].name: states[3],
                decoder.get_inputs()[5].name: states[4],
                decoder.get_inputs()[6].name: states[5],
                decoder.get_inputs()[7].name: encoder_out,
                decoder.get_inputs()[8].name: encoder_masks,
            },
  )
  states = new_states
  current = logits.argmax()

eos 3


# 查表

In [ ]:
tokens = [id2token[i] for i in ids]
print(tokens)

['▁A', 's', 'k', '▁not', '▁wh', 'at', '▁y', 'our', '▁co', 'un', 'tr', 'y', '▁can', '▁do', '▁for', '▁you', '.', '▁A', 's', 'k', '▁wh', 'at', '▁you', '▁can', '▁do', '▁for', '▁y', 'our', '▁co', 'un', 'tr', 'y', '.']


# 连起来所有的 tokens

In [ ]:
text = ''.join(tokens)
print(text)

▁Ask▁not▁what▁your▁country▁can▁do▁for▁you.▁Ask▁what▁you▁can▁do▁for▁your▁country.


In [ ]:
text = text.replace('▁', ' ')
print(text)

 Ask not what your country can do for you. Ask what you can do for your country.


# 再来听一下测试音频

In [ ]:
from IPython.display import Audio, display

display(Audio("./sherpa-onnx-nemo-canary-180m-flash-en-es-de-fr-int8/test_wavs/en.wav", autoplay=True))